<a href="https://colab.research.google.com/github/Safin44/my-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Lane:** Search performance / CTR opportunity ranking  
**Decision grain:** one pseudonymized content item (`client_hash_id` + `content_hash_id`) at the March 2026 monthly decision point.  
**Purpose:** create a leakage-aware, auditable feature frame from the gated FlyRank warehouse. All counts and scores are computed when this notebook runs; no results are fabricated here.

> **Access first:** request access to `FlyRank/internship-warehouse` and accept its data-use terms. In Colab, add a Secret named `HF_TOKEN` containing a Hugging Face **Read** token. Never put the token in a cell or commit it. Use only pseudonymized data and never attempt re-identification.


## 1. Contract — five plain-words answers

1. **What one row means:** one content item for one pseudonymized client, aggregated to a March 2026 decision snapshot. The underlying daily fact grain is one report date × client × content item.
2. **Tables:** `fact_content_daily_performance` for observed daily search/analytics signals. `dim_content` is not needed for this first feature frame; avoiding a join keeps the grain simple.
3. **Time window:** feature/decision month is **2026-03**. The future outcome window is **2026-04** (April's observed clicks compared with March's observed clicks). The final month, June 2026, is sealed and is not used to develop the label.
4. **What is ranked / proxy label:** rank content items for review using observable search-performance features. For the leakage demonstration only, the proxy label is `1` when April clicks are at least 20% lower than March clicks, among rows with non-zero March clicks. This is a directional proxy, not proof that a page needs an edit.
5. **Deliberately excluded:** client/content/hash IDs are used only for grouping and joins, never as predictive features. Raw identities, URLs, titles, and queries are not available and must not be inferred or reconstructed.

**Feature availability rule:** every model feature must be computable at the end of March, before the April outcome window begins.


In [1]:
# Setup and safe access. Works in Colab; locally, set HF_TOKEN as an environment variable.
%pip -q install duckdb pandas scikit-learn huggingface_hub

import os
import pandas as pd
import numpy as np
import duckdb
from sklearn.metrics import accuracy_score
from sklearn.dummy import DummyClassifier

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN not found. In Colab, add it under the key icon → Secrets. "
        "Accept the dataset access terms first; use a READ token."
    )

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
safe_token = hf_token.replace("'", "''")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{BASE}/fact_content_daily_performance/**/*.parquet"
FEATURE_MONTH = "2026-03"

# Schema inspection is setup, not one of the three verification queries.
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FACT}', hive_partitioning=true) LIMIT 0"
).df()
columns = set(schema["column_name"].str.lower())
print("Warehouse connection ready.")
print("Schema fields discovered:", len(columns))
print("Feature month:", FEATURE_MONTH)


Warehouse connection ready.
Schema fields discovered: 31
Feature month: 2026-03


## 2. Field buckets and feature availability

**Feature fields (maximum five):** monthly impressions, monthly clicks, CTR calculated as clicks / impressions, mean reported position, and monthly GA4 sessions when present. If a field is not in the released schema, the code stops rather than silently inventing it.

**Label:** April click decline proxy, calculated only for the leakage experiment and never retained in the honest feature matrix.

**Context / keys:** `report_date`, `client_hash_id`, `content_hash_id`, and the dataset's explicit search-availability boolean. Keys define the grain; they are not features.

**Excluded:** hash IDs as model inputs, any April/outcome-derived field from the honest feature matrix, and June `_sample` for label development.

- **Impressions:** knowable at the end of March because they are summed from March records.
- **Clicks:** knowable at the end of March because they are summed from March records.
- **CTR:** knowable at the end of March because it is computed from March clicks and impressions only.
- **Mean position:** knowable at the end of March because it is aggregated from March records only.
- **Sessions:** knowable at the end of March because they are summed from March records only, when this metric is present in the released table.


## 3. Three verification queries — run on March, not `_sample`

The three query results below are deliberately generated from the real gated warehouse at runtime:

1. **Grain:** duplicate count for `(report_date, client_hash_id, content_hash_id)`.
2. **Slice size and date span:** row count, minimum date, and maximum date for March 2026.
3. **Availability:** total March rows versus rows surviving the explicit `availability_column IS TRUE` filter.

The notebook detects the exact field names from the warehouse schema. If no availability boolean is present, it stops and asks for schema review; it does not substitute a non-boolean proxy.


In [2]:
# Resolve schema names once. This is schema discovery, not a verification query.
def pick(candidates, required=True):
    for candidate in candidates:
        if candidate.lower() in columns:
            return candidate.lower()
    if required:
        raise KeyError(f"Required field not found. Tried {candidates}. Available fields: {sorted(columns)}")
    return None

date_col = pick(["report_date", "date"])
client_col = pick(["client_hash_id"])
content_col = pick(["content_hash_id"])
impressions_col = pick(["gsc_impressions", "impressions"])
clicks_col = pick(["gsc_clicks", "clicks"])
position_col = pick(["gsc_avg_position", "gsc_position", "average_position", "position"])
sessions_col = pick(["ga4_sessions", "sessions"], required=False)
availability_col = pick([
    "gsc_available", "gsc_is_available", "has_gsc_data",
    "search_console_available", "is_gsc_available", "gsc_data_available"
])

print({
    "date": date_col, "client_key": client_col, "content_key": content_col,
    "impressions": impressions_col, "clicks": clicks_col,
    "position": position_col, "sessions": sessions_col,
    "availability_boolean": availability_col
})


{'date': 'report_date', 'client_key': 'client_hash_id', 'content_key': 'content_hash_id', 'impressions': 'gsc_impressions', 'clicks': 'gsc_clicks', 'position': 'gsc_avg_position', 'sessions': 'ga4_sessions', 'availability_boolean': 'gsc_data_available'}


In [3]:
# Verification query 1/3 — test the stated daily grain.
q1 = f'''
SELECT
  COUNT(*) AS march_rows,
  COUNT(*) - COUNT(DISTINCT ({date_col}, {client_col}, {content_col})) AS duplicate_grain_rows
FROM read_parquet('{FACT}', hive_partitioning=true)
WHERE CAST({date_col} AS DATE) >= DATE '2026-03-01'
  AND CAST({date_col} AS DATE) <  DATE '2026-04-01'
'''
grain_check = con.execute(q1).df()
display(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,duplicate_grain_rows
0,9841378,0


In [4]:
# Verification query 2/3 — row count and date span for the March slice.
q2 = f'''
SELECT
  COUNT(*) AS row_count,
  MIN(CAST({date_col} AS DATE)) AS first_date,
  MAX(CAST({date_col} AS DATE)) AS last_date
FROM read_parquet('{FACT}', hive_partitioning=true)
WHERE CAST({date_col} AS DATE) >= DATE '2026-03-01'
  AND CAST({date_col} AS DATE) <  DATE '2026-04-01'
'''
slice_check = con.execute(q2).df()
display(slice_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [5]:
# Verification query 3/3 — explicit boolean availability filter using IS TRUE.
q3 = f'''
SELECT
  COUNT(*) AS total_march_rows,
  COUNT(*) FILTER (WHERE {availability_col} IS TRUE) AS available_rows,
  COUNT(*) FILTER (WHERE {availability_col} IS NOT TRUE) AS unavailable_or_unknown_rows
FROM read_parquet('{FACT}', hive_partitioning=true)
WHERE CAST({date_col} AS DATE) >= DATE '2026-03-01'
  AND CAST({date_col} AS DATE) <  DATE '2026-04-01'
'''
availability_check = con.execute(q3).df()
display(availability_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_rows,available_rows,unavailable_or_unknown_rows
0,9841378,3611061,6230317


## 4. Build the five-feature frame + leakage trap

The feature frame is one row per client/content pair. March metrics are aggregated from March only. April clicks are joined separately to create the demonstration label; they are **not** allowed in the honest feature matrix.

The quick score is a simple train/test accuracy demonstration, not a production-quality evaluation. It uses a reproducible split and compares a deliberately leaked label-copy feature with the honest features. A near-perfect leaked score is expected because the added column literally contains the answer. The honest score is the only score to report as a meaningful baseline here.


In [6]:
# Build monthly aggregates for the March decision point and April outcome window.
# Keep identifiers only for grouping / joining, never as model features.
def monthly_aggregate(start_date, end_date, alias):
    session_expr = f"SUM(COALESCE({sessions_col}, 0))" if sessions_col else "CAST(NULL AS DOUBLE)"
    return f'''
      SELECT {client_col} AS client_id, {content_col} AS content_id,
             SUM(COALESCE({impressions_col}, 0))::DOUBLE AS impressions,
             SUM(COALESCE({clicks_col}, 0))::DOUBLE AS clicks,
             CASE WHEN SUM(COALESCE({impressions_col}, 0)) > 0
                  THEN SUM(COALESCE({clicks_col}, 0))::DOUBLE
                       / SUM(COALESCE({impressions_col}, 0))
                  ELSE NULL END AS ctr,
             AVG({position_col})::DOUBLE AS mean_position,
             {session_expr} AS sessions
      FROM read_parquet('{FACT}', hive_partitioning=true)
      WHERE CAST({date_col} AS DATE) >= DATE '{start_date}'
        AND CAST({date_col} AS DATE) < DATE '{end_date}'
        AND {availability_col} IS TRUE
      GROUP BY {client_col}, {content_col}
    '''

march_sql = monthly_aggregate("2026-03-01", "2026-04-01", "march")
april_sql = monthly_aggregate("2026-04-01", "2026-05-01", "april")

feature_sql = f'''
WITH march AS ({march_sql}),
     april AS ({april_sql})
SELECT
  m.client_id, m.content_id,
  m.impressions, m.clicks, m.ctr, m.mean_position, m.sessions,
  CASE WHEN m.clicks > 0 AND COALESCE(a.clicks, 0) <= 0.8 * m.clicks
       THEN 1 ELSE 0 END AS decline_label
FROM march m
LEFT JOIN april a
  ON m.client_id = a.client_id AND m.content_id = a.content_id
WHERE m.impressions > 0 AND m.clicks > 0
'''
feature_frame = con.execute(feature_sql).df()
if feature_frame.empty:
    raise RuntimeError("The March feature frame is empty after availability and volume filters; inspect the schema and filter.")
print("Feature frame shape:", feature_frame.shape)
display(feature_frame.drop(columns=["client_id", "content_id", "decline_label"]).head())
print("Label prevalence (April clicks at least 20% below March):")
display(feature_frame["decline_label"].value_counts(dropna=False).rename_axis("label").to_frame("rows"))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (68837, 8)


,impressions,clicks,ctr,mean_position,sessions
0,1423.0,1.0,0.000703,16.669625,0.0
1,2091.0,4.0,0.001913,9.837613,0.0
2,10239.0,31.0,0.003028,12.890116,0.0
3,14061.0,32.0,0.002276,4.467690,0.0
4,12022.0,7.0,0.000582,8.510640,0.0


Label prevalence (April clicks at least 20% below March):


,rows
label,
1,42299
0,26538


In [7]:
# Leakage experiment: compare honest features with one intentionally label-derived feature.
# No IDs or target-window measurements enter the honest feature matrix.
feature_cols = ["impressions", "clicks", "ctr", "mean_position", "sessions"]
X = feature_frame[feature_cols].copy()
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True)).fillna(0)
y = feature_frame["decline_label"].astype(int)

if y.nunique() < 2 or len(y) < 20:
    print("Not enough rows/classes for a meaningful split-score comparison. Leakage concept remains demonstrated in the explicit label-copy column below.")
else:
    from sklearn.model_selection import train_test_split
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )
    honest_model = make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")
    )
    honest_model.fit(X_train, y_train)
    honest_score = accuracy_score(y_test, honest_model.predict(X_test))

    # Deliberate trap: this column is the answer copied into the feature matrix.
    X_leaky = X.copy()
    X_leaky["LEAKED_label_copy"] = y.to_numpy()
    XL_train, XL_test, yL_train, yL_test = train_test_split(
        X_leaky, y, test_size=0.25, random_state=42, stratify=y
    )
    leaky_model = make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")
    )
    leaky_model.fit(XL_train, yL_train)
    leaky_score = accuracy_score(yL_test, leaky_model.predict(XL_test))

    score_comparison = pd.DataFrame([
        {"experiment": "Honest features only", "test_accuracy": honest_score},
        {"experiment": "INVALID: includes copied label", "test_accuracy": leaky_score},
    ])
    display(score_comparison)
    print(f"Honest score to retain: {honest_score:.3f}")
    print(f"Leaky score to reject: {leaky_score:.3f} — invalid because the feature reveals the target.")

# Delete the leakage column and retain only the honest five-feature frame.
if "X_leaky" in globals():
    del X_leaky
if "XL_train" in globals():
    del XL_train, XL_test, yL_train, yL_test
assert "LEAKED_label_copy" not in X.columns
print("Leakage feature removed. Honest feature columns:", list(X.columns))


,experiment,test_accuracy
0,Honest features only,0.536781
1,INVALID: includes copied label,1.000000


Honest score to retain: 0.537
Leaky score to reject: 1.000 — invalid because the feature reveals the target.
Leakage feature removed. Honest feature columns: ['impressions', 'clicks', 'ctr', 'mean_position', 'sessions']


## 5. Limitation

**Named limitation — uneven history and proxy-label ambiguity.** Client histories are unbalanced, and the April click-decline proxy can reflect seasonality, demand changes, or measurement variation rather than a fixable content problem. The result is suitable for prioritizing human review only; it is not causal evidence and should not be presented as a guaranteed ranking outcome.

## Self-check

- [x] Five plain-words contract answers are documented.
- [x] Exactly three verification queries are shown: grain, March count/date span, and availability (`IS TRUE`).
- [x] Five candidate features have an “available when?” explanation.
- [x] A label-copy leakage trap is added, compared, and removed from the honest feature matrix.
- [x] One named limitation is documented.
- [ ] **Run all cells after adding `HF_TOKEN`.** Confirm the query outputs and score table appear.
- [ ] Save/commit this executed notebook to `work/notebooks/w03_data_contract.ipynb` in your own public repo.
- [ ] Do not commit tokens, raw data, client-identifying data, or private queries.

**Important:** this notebook is prepared but cannot truthfully contain executed warehouse counts until run with your gated Hugging Face access. Do not submit fabricated counts or scores.
